In [36]:
# STEP 1: Imports and Setup

# Import necessary libraries for file handling, image processing, and deep learning
import os
import shutil
import random
from pathlib import Path
from PIL import Image
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

In [37]:
# STEP 2: Directory Setup

# Define the output directory where the processed dataset will be stored
output_dir = Path("/kaggle/working/wildlife_classification")

# Create subdirectories for training, validation, and test datasets
for split in ["train", "val", "test"]:
    (output_dir / split).mkdir(parents=True, exist_ok=True)

# Define paths to the original datasets from Kaggle and Roboflow
kaggle_base = Path("/kaggle/input/poaching-and-animal-detection-dataset/Project Dataset for Animal Prediction")
roboflow_base = Path("/kaggle/input/dataset-02")

In [38]:
# STEP 3: Labeling correctly.

# This block handles label normalization:
# - It maps various label forms (e.g., 'lion', 'Lion') to standardized category names like 'Wild Animal - Lion'.
# - The `infer_label_from_filename` function scans filenames and assigns them to the correct standardized label using the mapping.
# - It also creates two dictionaries:
#     1. `class_to_index`: Maps each unique standardized label to a numeric index (useful for model training).
#     2. `index_to_class`: Does the reverse – helps convert predictions back into human-readable class names.
# This ensures consistency across the dataset regardless of how labels were originally written in file names.

label_mapping = {
    # 🐾 Wild Animals
    'lion': 'Wild Animal - Lion',
    'Lion': 'Wild Animal - Lion',
    'tiger': 'Wild Animal - Tiger',
    'Tiger': 'Wild Animal - Tiger',
    'cheetah': 'Wild Animal - Cheetah',
    'Cheetah': 'Wild Animal - Cheetah',
    'leopard': 'Wild Animal - Leopard',
    'Jaguar': 'Wild Animal - Jaguar',
    'girafee': 'Wild Animal - Giraffe',
    'Giraffe': 'Wild Animal - Giraffe',
    'elephant': 'Wild Animal - Elephant',
    'Elephant': 'Wild Animal - Elephant',
    'zebra': 'Wild Animal - Zebra',
    'Zebra': 'Wild Animal - Zebra',
    'hippo': 'Wild Animal - Hippopotamus',
    'Hippopotamus': 'Wild Animal - Hippopotamus',
    'rhino': 'Wild Animal - Rhinoceros',
    'Rhinoceros': 'Wild Animal - Rhinoceros',
    'bear': 'Wild Animal - Bear',
    'Bear': 'Wild Animal - Bear',
    'Brown bear': 'Wild Animal - Bear',
    'Polar bear': 'Wild Animal - Bear',
    'fox': 'Wild Animal - Fox',
    'Fox': 'Wild Animal - Fox',
    'deer': 'Wild Animal - Deer',
    'Deer': 'Wild Animal - Deer',
    'pig': 'Wild Animal - Pig',
    'Pig': 'Wild Animal - Pig',
    'bull': 'Wild Animal - Bull',
    'Bull': 'Wild Animal - Bull',
    'Camel': 'Wild Animal - Camel',
    'camel': 'Wild Animal - Camel',
    'Cattle': 'Wild Animal - Bull',
    'Horse': 'Wild Animal - Horse',
    'Mule': 'Wild Animal - Mule',
    'Goat': 'Wild Animal - Goat',
    'Kangaroo': 'Wild Animal - Kangaroo',
    'Koala': 'Wild Animal - Koala',
    'Otter': 'Wild Animal - Otter',
    'Raccoon': 'Wild Animal - Raccoon',
    'Red panda': 'Wild Animal - Red Panda',
    'Hedgehog': 'Wild Animal - Hedgehog',
    'Mouse': 'Wild Animal - Mouse',
    'Lynx': 'Wild Animal - Lynx',
    'Hamster': 'Wild Animal - Hamster',
    'Squirrel': 'Wild Animal - Squirrel',
    'Panda': 'Wild Animal - Panda',

    # 🧍 Human
    'poacher': 'Human - Poacher',
    'Poacher': 'Human - Poacher',
    'ranger': 'Human - Ranger',
    'Ranger': 'Human - Ranger',

    # 🔫 Weapon
    'weapon': 'Weapon',
    'Weapon': 'Weapon',

    # 🐒 Monkey
    'monkey': 'Wild Animal - Monkey',
    'Monkey': 'Wild Animal - Monkey',

    # 🐦 Birds
    'Turkey': 'Bird - Turkey',
    'turkey': 'Bird - Turkey',
    'Eagle': 'Bird - Eagle',
    'Owl': 'Bird - Owl',
    'Parrot': 'Bird - Parrot',
    'Sparrow': 'Bird - Sparrow',
    'Canary': 'Bird - Canary',
    'Goose': 'Bird - Goose',
    'Chicken': 'Bird - Chicken',
    'Duck': 'Bird - Duck',
    'Raven': 'Bird - Raven',
    'Magpie': 'Bird - Magpie',
    'Woodpecker': 'Bird - Woodpecker',
    'Swan': 'Bird - Swan',
    'Ostrich': 'Bird - Ostrich',

    # 🐛 Insects
    'Butterfly': 'Insect - Butterfly',
    'Moths and butterflies': 'Insect - Butterfly',
    'Caterpillar': 'Insect - Caterpillar',
    'Centipede': 'Insect - Centipede',
    'Ladybug': 'Insect - Ladybug',
    'Tick': 'Insect - Tick',
    'Scorpion': 'Insect - Scorpion',
    'Spider': 'Insect - Spider',
    'Worm': 'Insect - Worm',

    # 🐟 Marine Life
    'Fish': 'Marine Animal - Fish',
    'Shark': 'Marine Animal - Shark',
    'Sea turtle': 'Marine Animal - Sea Turtle',
    'Turtle': 'Marine Animal - Turtle',
    'Tortoise': 'Marine Animal - Tortoise',
    'Seahorse': 'Marine Animal - Seahorse',
    'Crab': 'Marine Animal - Crab',
    'Shrimp': 'Marine Animal - Shrimp',
    'Squid': 'Marine Animal - Squid',
    'Jellyfish': 'Marine Animal - Jellyfish',
    'Whale': 'Marine Animal - Whale',
    'Sea lion': 'Marine Animal - Sea Lion',
    'Harbor seal': 'Marine Animal - Seal',
    'Starfish': 'Marine Animal - Starfish',
    'Goldfish': 'Marine Animal - Fish',

    # 🦎 Reptiles & Amphibians
    'Snake': 'Reptile - Snake',
    'Lizard': 'Reptile - Lizard',
    'Frog': 'Amphibian - Frog',

    # ⚠️ Fallback
    'animal': 'Wild Animal - Unknown',
    'dog': 'Wild Animal - Dog',  # domesticated
    'Cat': 'Wild Animal - Cat',  # if present
    'unknown': 'Wild Animal - Unknown'
}

# Normalize label from filename
def infer_label_from_filename(filename):
    name = filename.lower()
    for key in label_mapping:
        if key.lower() in name:
            return label_mapping[key]
    return None
    
# Create class list from unique mapped values
unique_classes = sorted(set(label_mapping.values()))
class_to_index = {name: i for i, name in enumerate(unique_classes)}
index_to_class = {v: k for k, v in class_to_index.items()}

# Example of using it:
print("Total Classes:", len(class_to_index))
for i, name in enumerate(unique_classes):
    print(f"{i}: {name}")

Total Classes: 76
0: Amphibian - Frog
1: Bird - Canary
2: Bird - Chicken
3: Bird - Duck
4: Bird - Eagle
5: Bird - Goose
6: Bird - Magpie
7: Bird - Ostrich
8: Bird - Owl
9: Bird - Parrot
10: Bird - Raven
11: Bird - Sparrow
12: Bird - Swan
13: Bird - Turkey
14: Bird - Woodpecker
15: Human - Poacher
16: Human - Ranger
17: Insect - Butterfly
18: Insect - Caterpillar
19: Insect - Centipede
20: Insect - Ladybug
21: Insect - Scorpion
22: Insect - Spider
23: Insect - Tick
24: Insect - Worm
25: Marine Animal - Crab
26: Marine Animal - Fish
27: Marine Animal - Jellyfish
28: Marine Animal - Sea Lion
29: Marine Animal - Sea Turtle
30: Marine Animal - Seahorse
31: Marine Animal - Seal
32: Marine Animal - Shark
33: Marine Animal - Shrimp
34: Marine Animal - Squid
35: Marine Animal - Starfish
36: Marine Animal - Tortoise
37: Marine Animal - Turtle
38: Marine Animal - Whale
39: Reptile - Lizard
40: Reptile - Snake
41: Weapon
42: Wild Animal - Bear
43: Wild Animal - Bull
44: Wild Animal - Camel
45: Wil

In [39]:
# Step 4: Organize the Kaggle dataset into proper folder structures for model training.
# - This function takes the base dataset directory and the desired split name ('train', 'test').
# - It reads through each class folder, normalizes the class name using the label_mapping dictionary,
#   and copies image files into the appropriate subfolder (under train/val/test).
# - If `val_split=True`, it performs an 80-20 split on the images to create both train and val folders manually.
# - Ensures all images are saved under consistent class labels for model training.

def organize_kaggle_split(base_dir, split_name, val_split=False):
    for class_dir in os.listdir(base_dir):
        source = base_dir / class_dir
        if not source.is_dir():
            continue
        label = label_mapping.get(class_dir, None)
        if not label:
            continue
        files = [f for f in os.listdir(source) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if val_split:
            random.shuffle(files)
            split_idx = int(len(files) * 0.8)
            files_train = files[:split_idx]
            files_val = files[split_idx:]
            for file in files_train:
                shutil.copy(source / file, output_dir / "train" / label / file)
            for file in files_val:
                shutil.copy(source / file, output_dir / "val" / label / file)
        else:
            for file in files:
                shutil.copy(source / file, output_dir / split_name / label / file)

In [40]:
# Step 5: Organize Roboflow flat folder structure into train/val/test splits

def organize_roboflow_flat(src_dir, split_name):
    for file in os.listdir(src_dir):
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            label = infer_label_from_filename(file)  # Assumes you have this function defined
            if not label:
                continue
            # Ensure the destination folder exists
            dest_path = output_dir / split_name / label
            dest_path.mkdir(parents=True, exist_ok=True)
            shutil.copy(src_dir / file, dest_path / file)

# Step 5.1: Create label subfolders in advance for all splits
for split in ["train", "val", "test"]:
    for label in unique_classes:  # unique_classes should be a list of all label names
        (output_dir / split / label).mkdir(parents=True, exist_ok=True)

# Step 5.2: Organize both Kaggle and Roboflow datasets
organize_kaggle_split(kaggle_base / "train", "train", val_split=True)
organize_kaggle_split(kaggle_base / "test", "test", val_split=False)
organize_roboflow_flat(roboflow_base / "train/images", "train")
organize_roboflow_flat(roboflow_base / "valid/images", "val")
organize_roboflow_flat(roboflow_base / "test/images", "test")

print("✅ Dataset organized.")

✅ Dataset organized.


In [ ]:
# STEP 6: Dataset & DataLoaders

# 6.1: Remove any empty class directories (caused by missing data or filtering)
def remove_empty_class_dirs(split_path):
    for class_dir in split_path.iterdir():
        if class_dir.is_dir() and not any(class_dir.glob("*")):
            print(f"Removing empty class directory: {class_dir}")
            shutil.rmtree(class_dir)

for split in ["train", "val", "test"]:
    remove_empty_class_dirs(output_dir / split)

# 6.2: Define transformations for training (with augmentations) and validation/testing (without augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),                         # Resize all images to 224x224
    transforms.RandomHorizontalFlip(),                     # Random horizontal flip
    transforms.RandomVerticalFlip(),                       # Random vertical flip
    transforms.RandomRotation(15),                         # Random rotation up to 15 degrees
    transforms.ColorJitter(brightness=0.2,                 # Random brightness adjustment
                           contrast=0.2, saturation=0.2),  # Random contrast & saturation
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),  # Random affine transformation
    transforms.ToTensor(),                                 # Convert image to PyTorch tensor
    transforms.Normalize([0.485, 0.456, 0.406],             # Normalize using ImageNet mean & std
                         [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),                         # Resize for consistency
    transforms.ToTensor(),                                 # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406],             # Normalize
                         [0.229, 0.224, 0.225])
])

# 6.3: Create datasets using ImageFolder (folder names are treated as class labels)
train_dataset = datasets.ImageFolder(output_dir / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(output_dir / "val", transform=val_test_transform)
test_dataset = datasets.ImageFolder(output_dir / "test", transform=val_test_transform)

# 6.4: Create DataLoaders for batch processing and shuffling (for training)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# 6.5: Confirmation message
print("✅ Datasets loaded.")
print("Classes:", train_dataset.classes)

# **ResNet34**

In [ ]:
# STEP 7: Model Setup and Training Preparation

from torch.cuda.amp import GradScaler

# 7.1: Check if a GPU is available, else fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 7.2: Load pre-trained ResNet-34 model from torchvision
model = models.resnet34(pretrained=True)

# 7.3: Modify the final fully connected layer (fc) to match the number of classes in the dataset
model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))

# 7.4: Move the model to the selected device (GPU or CPU)
model = model.to(device)

# 7.5: Define the loss function as CrossEntropyLoss for classification tasks
criterion = nn.CrossEntropyLoss()

# 7.6: Initialize the optimizer (Adam) with a learning rate of 0.0005
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# 7.7: Set up a learning rate scheduler to reduce the learning rate if the validation metric plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# 7.8: Set up mixed precision training for faster training and memory efficiency using GradScaler
scaler = GradScaler()


In [ ]:
from torch.amp import autocast  # Import from torch.amp instead of torch.cuda.amp

# 8.1: Evaluate the model on the validation/test set
def evaluate_model(loader):
    model.eval()  # Set the model to evaluation mode (disables dropout and batch normalization)
    correct, total = 0, 0
    
    with torch.no_grad():  # Disable gradient calculation for evaluation (saves memory and computations)
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)  # Move data to the device (GPU/CPU)
            outputs = model(images)  # Get model predictions
            _, preds = torch.max(outputs, 1)  # Get the predicted class (index with max probability)
            correct += (preds == labels).sum().item()  # Count correct predictions
            total += labels.size(0)  # Count total labels
            
    # Return accuracy as a percentage
    return 100 * correct / total

# 8.2: Train the model for a specified number of epochs
def train_model(epochs=30):
    for epoch in range(epochs):
        model.train()  # Set the model to training mode (enables dropout and batch normalization)
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)  # Move data to the device (GPU/CPU)
            optimizer.zero_grad()  # Clear previous gradients

            with autocast('cuda'):  # Enable mixed precision (use float16 for faster computation)
                outputs = model(images)  # Get model predictions
                loss = criterion(outputs, labels)  # Compute the loss

            scaler.scale(loss).backward()  # Scale the loss and backpropagate
            scaler.step(optimizer)  # Update model weights
            scaler.update()  # Update the scaler

            running_loss += loss.item()  # Accumulate the loss for reporting
            _, preds = torch.max(outputs, 1)  # Get the predicted class
            correct += (preds == labels).sum().item()  # Count correct predictions
            total += labels.size(0)  # Count total labels

        # Evaluate model on validation set and adjust learning rate
        val_acc = evaluate_model(val_loader)
        scheduler.step(val_acc)  # Reduce learning rate if validation accuracy plateaus

        # Print training stats for the current epoch
        print(f"Epoch {epoch+1} | Loss: {running_loss:.4f} | Train Acc: {100*correct/total:.2f}% | Val Acc: {val_acc:.2f}%")

# 8.3: Start the training process
print("\n Training started...")
train_model(epochs=30)  # Train for 30 epochs

# 8.4: Evaluate model on the test set after training
print("\n Final Test Accuracy:", evaluate_model(test_loader), "%")


In [ ]:
# Function to evaluate the model and generate a classification report and confusion matrix
def evaluate_and_report(model, dataloader, class_names):
    # 1. Set the model to evaluation mode
    model.eval()  # Disables dropout and batch normalization layers
    
    # 2. Initialize lists to store predictions and labels for the entire dataset
    all_preds, all_labels = [], []

    # 3. Iterate through the data in the dataloader without computing gradients
    with torch.no_grad():  # Disables gradient computation, saving memory during inference
        for images, labels in dataloader:
            images = images.to(device)  # Move images to the device (GPU/CPU)
            outputs = model(images)  # Get the model's predictions
            _, preds = torch.max(outputs, 1)  # Get the index of the maximum output (predicted class)
            
            # 4. Append predictions and true labels to the respective lists
            all_preds.extend(preds.cpu().numpy())  # Convert predictions to numpy and add to list
            all_labels.extend(labels.cpu().numpy())  # Convert true labels to numpy and add to list

    # 5. Print a detailed classification report using sklearn's classification_report
    print("\n Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))  # Precision, Recall, F1-score

    # 6. Compute the confusion matrix for the model's predictions
    cm = confusion_matrix(all_labels, all_preds)  # Generate confusion matrix from true and predicted labels
    
    # 7. Visualize the confusion matrix using Seaborn's heatmap
    plt.figure(figsize=(10, 8))  # Set the size of the confusion matrix plot
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.title(" Confusion Matrix")  # Set the title of the plot
    plt.xlabel("Predicted")  # Set the x-axis label
    plt.ylabel("True")  # Set the y-axis label
    plt.show()  # Display the plot

# 8. Evaluate the model on the test set and print the classification report and confusion matrix
evaluate_and_report(model, test_loader, train_dataset.classes)  # Test the model and display the results


In [ ]:
from torchvision import transforms as T
import matplotlib.patches as patches

# Function to predict the class and draw a rectangle around the image
def predict_and_draw(img_path):
    # 1. Open the image and convert to RGB format
    image = Image.open(img_path).convert("RGB")
    
    # 2. Apply the same transformations (resize, normalize, etc.) as during training
    img_tensor = val_test_transform(image).unsqueeze(0).to(device)  # Convert to tensor and add batch dimension
    
    # 3. Set the model to evaluation mode
    model.eval()
    
    # 4. Perform inference (no gradient calculation for evaluation)
    with torch.no_grad():  # No gradients are needed for inference
        output = model(img_tensor)  # Pass the image tensor through the model
        _, pred = torch.max(output, 1)  # Get the predicted class index (the one with highest score)
        label = train_dataset.classes[pred.item()]  # Get the class name based on the predicted index

    # 5. Display the image with a rectangle and label
    plt.figure(figsize=(5, 5))  # Set the size of the display window
    plt.imshow(image)  # Display the image
    
    ax = plt.gca()  # Get the current axes of the plot
    # 6. Add a red rectangle around the image (with some padding)
    rect = patches.Rectangle((10, 10), image.size[0] - 20, image.size[1] - 20, 
                             linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)  # Add the rectangle to the plot

    # 7. Add a label with the predicted class name
    plt.text(15, 25, label, color='white', backgroundcolor='red', fontsize=12)
    
    plt.axis('off')  # Hide the axes (we just want to see the image)
    plt.title(f"Prediction: {label}")  # Set the title of the plot with the predicted label
    plt.show()  # Display the image and the plot

# Example usage:
predict_and_draw("/kaggle/input/test-img/test_poacher.jpeg")


In [ ]:
# Define the path where the model will be saved
model_path = "/kaggle/working/wildlife_resnet_model.pth"

# Save the model's state_dict to the specified file path
torch.save(model.state_dict(), model_path)

# Print a confirmation message with the saved model path
print(f"✅ Model saved to: {model_path}")


In [ ]:
import shutil

# Define the path where the model is saved
model_path = "/kaggle/working/wildlife_resnet_model.pth"

# Create a zip archive of the saved model
shutil.make_archive("/kaggle/working/wildlife_model", 'zip', root_dir="/kaggle/working", base_dir="wildlife_resnet_model.pth")

# Print a success message
print("✅ Model zipped successfully.")


In [ ]:
import os
from IPython.display import FileLink

# Define the path to the saved model
model_path = "/kaggle/working/wildlife_resnet_model.pth"

# Check if the model file exists at the specified path
if os.path.exists(model_path):
    # If it exists, display a clickable download link
    display(FileLink(model_path))
else:
    # If the model file doesn't exist, print an error message
    print(" Model file not found. Check the path.")
